# GTEx production quality control

This report reads Snakemake production artifacts only. It does not build the GTEx
FBM/expression matrix, fit models, or run CLAMP.

💡 **Environment:** `clamp-analyses`

## Libraries

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import yaml
from pyprojroot import here

## Settings

In [ ]:
ROOT = Path(here())
with open(ROOT / 'workflow/config/gtex.yaml') as handle:
    CONFIG = yaml.safe_load(handle)['gtex']
PROD = ROOT / CONFIG['paths']['production']
OUT = PROD / 'qc'
OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
def csv_shape(path):
    # Count rows as bytes so QC never parses/materializes large numeric model matrices.
    with open(path, 'rb') as handle:
        header_line = handle.readline()
        n_rows = 0
        while chunk := handle.read(8 * 1024 * 1024):
            n_rows += chunk.count(b'\n')
    n_cols = header_line.count(b',')
    return n_rows, n_cols

def b_shape(path):
    if path.suffix == '.pkl':
        return pd.read_pickle(path).shape
    return csv_shape(path)

METHOD_B_PATHS = {
    'CLAMPbase': 'CLAMPbase/B.csv',
    'CLAMPfull': 'CLAMPfull/B.csv',
    'PLIER': 'PLIER/B.csv',
    'PCA': 'PCA/gtex_pca_B.pkl',
    'ICA': 'ICA/gtex_ica_B.pkl',
    'NMF': 'NMF/gtex_nmf_B.pkl',
    'flashier': 'flashier/gtex_B.csv',
    'MOFA_FLEX_PRIOR': 'MOFA_FLEX_PRIOR/B_matrix.csv',
    'GSS': 'GSS/gtex_B.csv',
}

model_rows = []
for method, rel_path in METHOD_B_PATHS.items():
    b_path = PROD / rel_path
    complete = b_path.exists()
    n_lvs, n_samples = b_shape(b_path) if complete else (None, None)
    model_rows.append({
        'method': method, 'complete': complete,
        'n_LVs': n_lvs, 'n_samples': n_samples,
    })
models_df = pd.DataFrame(model_rows)

rank = int(pd.read_csv(PROD / 'CLAMP_K_gtex.csv')['CLAMP_K_gtex'].iloc[0])
input_genes, input_samples = csv_shape(PROD / 'df_gtex_fbm_filt.csv')

summary_df = pd.DataFrame([{
    'rank': rank,
    'input_genes': input_genes,
    'input_samples': input_samples,
    'models_complete': int(models_df['complete'].sum()),
    'models_expected': len(models_df),
}])

summary_df.to_csv(OUT / 'model_building_qc.csv', index=False)
models_df.to_csv(OUT / 'model_matrix_qc.csv', index=False)
summary_df


In [ ]:
models_df


In [ ]:
complete_df = models_df[models_df['complete']].sort_values('n_LVs')

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.barh(complete_df['method'], complete_df['n_LVs'])
ax.set_xlabel('LVs recovered (B matrix rows)')
ax.set_title(f"GTEx model building (rank K={rank})")
fig.tight_layout()
fig.savefig(OUT / 'model_building_qc.png', dpi=160, bbox_inches='tight')
plt.show()
